## Setup

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from src.py_src.models import SolarfallFeatureEngineer
import urllib.request

In [ ]:
load_dotenv()

EVENTS_TREATED_PATH = os.getenv('EVENTS_TREATED_PATH')
XRAY_TREATED_PATH = os.getenv('XRAY_TREATED_PATH')
MAG_TREATED_GLOBAL_PATH = os.getenv('MAG_TREATED_GLOBAL_PATH')
MAG_TREATED_BY_REGION_PATH = os.getenv('MAG_TREATED_BY_REGION_PATH')

SLIDED_PATH = os.getenv('SLIDED_PATH')

## Read Files

In [ ]:
print("Carregando bases tratadas...")

# 1. Leitura com materialização explícita na memória (quebra o vínculo com o PyArrow)
df_events = pd.read_parquet(os.path.join(EVENTS_TREATED_PATH, "treated_events.parquet")).copy()
df_xray = pd.read_parquet(os.path.join(XRAY_TREATED_PATH, "treated_xray.parquet")).copy()
df_mag_global = pd.read_parquet(os.path.join(MAG_TREATED_GLOBAL_PATH, "treated_mag_global.parquet")).copy()
df_mag_regional = pd.read_parquet(os.path.join(MAG_TREATED_BY_REGION_PATH, "treated_mag_regional.parquet")).copy()

# 2. Conversão segura (agora que somos donos absolutos da memória do DataFrame)
df_events['peak_time'] = pd.to_datetime(df_events['peak_time'], utc=True)
df_events['active_region_no'] = pd.to_numeric(df_events['active_region_no'], errors='coerce')

df_xray['time'] = pd.to_datetime(df_xray['time'], utc=True)

df_mag_global['T_REC_round'] = pd.to_datetime(df_mag_global['T_REC_round'], utc=True)

df_mag_regional['T_REC_round'] = pd.to_datetime(df_mag_regional['T_REC_round'], utc=True)
df_mag_regional['REGION_ID'] = pd.to_numeric(df_mag_regional['REGION_ID'], errors='coerce')

In [ ]:
# Confere se as alterações foram realizadas, mesmo com o warning falso positivo do pandas
print(df_events['peak_time'].dtype)

## Generate Sliding Windows

In [ ]:
feature_engineer = SolarfallFeatureEngineer(freq='12min')

print("Gerando Família A (X-Ray)...")
features_a = feature_engineer.generate_family_a_xray(df_xray)

print("Gerando Família B (Mag Global)...")
features_b = feature_engineer.generate_family_b_mag_global(df_mag_global)

print("Gerando Família C (Mag Regional)...")
features_c = feature_engineer.generate_family_c_mag_regional(df_mag_regional)

In [ ]:
features_a.head()

In [ ]:
features_b.head()

In [ ]:
features_c.head()

### 📝 Nota Metodológica: Resolvendo a Colisão de Hash (Truncamento NOAA)

Ao mapear as Regiões Ativas (ARs), enfrentamos uma divergência histórica: o catálogo GOES/SWPC frequentemente trunca os números das regiões (ex: registrando `11158` como `1158`), enquanto o Stanford JSOC (HARP) mantém os 5 dígitos.

Para garantir o acoplamento correto das explosões aos dados magnéticos, nossa função injeta matematicamente a versão truncada de 4 dígitos (`% 10000`) na lista de ARs válidos para cada HARP.

**Salvaguarda contra Colisão Espacial:**
Embora a região `AR 1158` (década de 1980) e a `AR 11158` (2011) possuam o mesmo "ID truncado", isso **não causa data leakage ou mistura de épocas**. A nossa função de acoplamento (`append_targets`) aplica uma **salvaguarda temporal hierárquica**: ela primeiro isola uma janela estrita de 24 horas usando busca binária (`np.searchsorted`). Como essas regiões estão separadas por 30 anos, é fisicamente impossível que uma explosão da década de 80 caia na janela de 24 horas de um magnetograma de 2011. O filtro espacial (`np.isin`) atua de forma perfeitamente segura dentro desse micro-recorte temporal.

In [ ]:
def load_harp_to_noaa_map():
    """
    Descarrega o mapeamento oficial HARP -> NOAA do JSOC.
    Retorna um dicionário onde a chave é o HARPNUM (int) e o valor é uma lista de NOAA ARs (list of ints).
    INCLUI TOLERÂNCIA A TRUNCAMENTO DO GOES (4 e 5 dígitos).
    """
    url = "http://jsoc.stanford.edu/doc/data/hmi/harpnum_to_noaa/all_harps_with_noaa_ars.txt"
    harp_to_noaa = {}
    print("A descarregar a tabela oficial de mapeamento HARP-NOAA de Stanford...")

    try:
        # Usa um User-Agent por boas práticas de requisição
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})

        with urllib.request.urlopen(req) as response:
            for line in response:
                line = line.decode('utf-8').strip()

                # Ignora linhas vazias ou o cabeçalho
                if not line or line.startswith('HARPNUM'):
                    continue

                parts = line.split()
                if len(parts) >= 2:
                    harpnum = int(parts[0])
                    noaas = []

                    for part in parts[1:]:
                        for n in part.split(','):
                            if n.isdigit():
                                noaa_val = int(n)
                                noaas.append(noaa_val)

                                # Se a NOAA tem 5 dígitos (ex: 11067), adicionamos a versão de 4 dígitos (1067)
                                if noaa_val >= 10000:
                                    noaas.append(noaa_val % 10000)

                    harp_to_noaa[harpnum] = noaas

        print(f"Mapeamento concluído com sucesso! {len(harp_to_noaa)} HARPs mapeados.")
    except Exception as e:
        print(f"Erro ao descarregar o mapeamento: {e}")

    return harp_to_noaa

harp_to_noaa_map = load_harp_to_noaa_map()

In [ ]:
print("Acoplando Targets (24h)...")

slided_a = feature_engineer.append_targets(features_a, df_events, time_col='time', window_hours=24, is_regional=False)
slided_b = feature_engineer.append_targets(features_b, df_events, time_col='T_REC_round', window_hours=24, is_regional=False)

slided_c = feature_engineer.append_targets(features_c, df_events, time_col='T_REC_round', window_hours=24, is_regional=True, regional_col_feature='REGION_ID', harp_to_noaa_map=harp_to_noaa_map)

## Showing DFs

In [ ]:
slided_a.head()

In [ ]:
slided_a['target_class_in_24h'].mean()

In [ ]:
slided_b.head()

In [ ]:
slided_b['target_class_in_24h'].mean()

In [ ]:
slided_c.head()

In [ ]:
slided_c['target_class_in_24h'].mean()

## Exporting

In [ ]:
os.makedirs(SLIDED_PATH, exist_ok=True)

slided_a.to_parquet(os.path.join(SLIDED_PATH, "xray_slided.parquet"), index=False)
slided_b.to_parquet(os.path.join(SLIDED_PATH, "mag_global_slided.parquet"), index=False)
slided_c.to_parquet(os.path.join(SLIDED_PATH, "mag_regional_slided.parquet"), index=False)